# Run TGNNExplainer

Run this notebook for the selected `DATASET` / `MODEL` combo.

This notebook:
- Runs the official `TGNNExplainer` / `SubgraphXTG` implementation from the vendored TGNNExplainer submodule.
- Loads the matching checkpoint and explain index for the chosen combo.
- Saves official run artifacts under `resources/results/official_tgnnexplainer/` and finishes with the shared one-spot metrics summary.

Usage:
- Change `DATASET` and `MODEL` in the first code cell when needed.
- Run the notebook top to bottom.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

_PROJECT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next((p for p in _PROJECT_CANDIDATES if (p / "I_explainer_benchmark").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root from the current working directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from I_explainer_benchmark.src.notebooks.explainer_notebook_setup import (
    initialize_explainer_notebook,
    prepare_explainer_run,
)

# Select the dataset / model combo here.
DATASET = "simulate_v1"
MODEL = "tgn"

# Shared notebook setup. Leave the rest unchanged.
BOOT = initialize_explainer_notebook("05_tgnnexplainer.ipynb", dataset=DATASET, model=MODEL, start=PROJECT_ROOT)
NB = BOOT.env
CFG = BOOT.config
SETTINGS = BOOT.settings
BENCHMARK_ROOT = BOOT.bench_root
REPO_ROOT = BOOT.repo_root
TGNN_VENDOR = BOOT.paths["TGNN_VENDOR"]


In [2]:
import torch
import yaml
from I_explainer_benchmark.src.notebooks.notebook_helpers import (
    resolve_tgnnexplainer_vendor_dataset,
    set_global_seed,
)

MODEL = str(MODEL).strip().lower()
SUPPORTED_MODELS = set(CFG["supported_model_types"])
if MODEL not in SUPPORTED_MODELS:
    raise ValueError(f"Unsupported MODEL={MODEL!r}. Choose one of {sorted(SUPPORTED_MODELS)}")

CTX = prepare_explainer_run("05_tgnnexplainer.ipynb", dataset=DATASET, model=MODEL, start=PROJECT_ROOT)

DATASET_NAME = CTX.dataset
MODEL_NAME = CTX.model
SEED = int(CFG["seed"])
set_global_seed(SEED)

EXPLAIN_START = int(SETTINGS.get("explain_start", 0))
N_EVENTS = int(SETTINGS.get("n_events", 30))
USE_NAVIGATOR = bool(SETTINGS.get("use_navigator", True))
NAVIGATOR_TYPE = str(SETTINGS.get("navigator_type", "pg"))
PG_POSITIVE = bool(SETTINGS.get("pg_positive", True))
RUN_PREDICTION_PROFILE = bool(SETTINGS.get("run_prediction_profile", False))
RUN_EXTENDED_METRICS = bool(SETTINGS.get("run_extended_metrics", False))
RETRAIN_NAVIGATOR = bool(SETTINGS.get("retrain_navigator", False))

VENDOR_DATASET_NAME = resolve_tgnnexplainer_vendor_dataset(DATASET_NAME)
if VENDOR_DATASET_NAME != DATASET_NAME:
    print(f"Using TGNNExplainer vendor config alias: {DATASET_NAME} -> {VENDOR_DATASET_NAME}")


def _load_vendor_param(config_group: str, config_name: str, dataset_name: str) -> tuple[dict, Path]:
    cfg_path = TGNN_VENDOR / "benchmarks" / "xgraph" / "config" / config_group / config_name
    if not cfg_path.exists():
        raise FileNotFoundError(f"Missing vendored config: {cfg_path}")
    payload = yaml.safe_load(cfg_path.read_text()) or {}
    params = payload.get("param") or {}
    vendor_dataset = resolve_tgnnexplainer_vendor_dataset(dataset_name, available=params.keys())
    try:
        dataset_param = dict(params[vendor_dataset])
    except KeyError as exc:
        raise KeyError(
            f"Missing vendored config params in {cfg_path.name!r} for dataset={dataset_name!r}."
        ) from exc
    return dataset_param, cfg_path


p, MODEL_CFG_PATH = _load_vendor_param("models", f"{MODEL_NAME}.yaml", DATASET_NAME)
subgraphx_param, EXPLAINER_CFG_PATH = _load_vendor_param("explainers", "subgraphx_tg.yaml", DATASET_NAME)

THRESHOLD_NUM = int(SETTINGS.get("threshold_num", subgraphx_param["threshold_num"]))
ROLLOUT = int(SETTINGS.get("rollout", subgraphx_param["rollout"]))
MIN_ATOMS = int(SETTINGS.get("min_atoms", subgraphx_param["min_atoms"]))
C_PUCT = float(SETTINGS.get("c_puct", subgraphx_param["c_puct"]))
TRAIN_EPOCHS = int(SETTINGS.get("train_epochs", subgraphx_param["train_epochs"]))
REG_COEFS = list(SETTINGS.get("reg_coefs", subgraphx_param["reg_coefs"]))
BATCH_SIZE = int(SETTINGS.get("batch_size", subgraphx_param["batch_size"]))
LR = float(SETTINGS.get("lr", subgraphx_param["lr"]))
EXPLANATION_LEVEL = str(SETTINGS.get("explanation_level", subgraphx_param["explanation_level"]))

CKPT_PATH = CTX.checkpoint_path
EXPLAIN_INDEX_PATH = CTX.explain_index_path
DEVICE = CTX.device
RESULTS_DIR = BENCHMARK_ROOT / "resources" / "results" / "official_tgnnexplainer"
MCTS_SAVED_DIR = RESULTS_DIR / "mcts_saved_dir"
EXPLAINER_CKPT_DIR = BENCHMARK_ROOT / "resources" / "models" / "tgnnexplainer" / DATASET_NAME / MODEL_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MCTS_SAVED_DIR.mkdir(parents=True, exist_ok=True)
EXPLAINER_CKPT_DIR.mkdir(parents=True, exist_ok=True)

print("DATASET:", DATASET_NAME)
print("MODEL  :", MODEL_NAME)
print("CKPT   :", CKPT_PATH)
print("DEVICE :", DEVICE)


DATASET: simulate_v1
MODEL  : tgn
CKPT   : /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/models/simulate_v1/tgn/tgn_simulate_v1_best.pth
DEVICE : mps


In [3]:
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import random 
from tgnnexplainer.xgraph.dataset.tg_dataset import load_explain_idx, load_tg_dataset
from I_explainer_benchmark.src.data.io import load_processed_dataset
from I_explainer_benchmark.src.explainers.adapters._tgnn_dataframe_compat import install_tgnn_dataframe_compat
from I_explainer_benchmark.src.notebooks.notebook_helpers import forced_target_event_ids_for_combo as _forced_target_event_ids_for_combo
from tgnnexplainer.xgraph.dataset.utils_dataset import construct_tgat_neighbor_finder
from tgnnexplainer.xgraph.evaluation.metrics_tg import EvaluatorMCTSTG
from tgnnexplainer.xgraph.method.navigators import DotProductNavigator, MLPNavigator, PGNavigator
from tgnnexplainer.xgraph.method.subgraphx_tg import SubgraphXTG
from tgnnexplainer.xgraph.models.ext.tgat.module import TGAN
from tgnnexplainer.xgraph.models.ext.tgn.model.tgn import TGN
from tgnnexplainer.xgraph.models.ext.tgn.utils.data_processing import compute_time_statistics

explainer_src = Path(inspect.getfile(SubgraphXTG)).resolve()
evaluator_src = Path(inspect.getfile(EvaluatorMCTSTG)).resolve()
assert "/submodules/explainer/tgnnexplainer/" in str(explainer_src).replace("\\", "/").lower()
assert "/submodules/explainer/tgnnexplainer/" in str(evaluator_src).replace("\\", "/").lower()

print("Official SubgraphXTG source:", explainer_src)
print("Official Evaluator source :", evaluator_src)


Official SubgraphXTG source: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/submodules/explainer/tgnnexplainer/tgnnexplainer/xgraph/method/subgraphx_tg.py
Official Evaluator source : /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/submodules/explainer/tgnnexplainer/tgnnexplainer/xgraph/evaluation/metrics_tg.py


In [4]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

_events_bundle = load_processed_dataset(DATASET_NAME)
install_tgnn_dataframe_compat(events=_events_bundle["interactions"], dataset_name=DATASET_NAME)

events, edge_feats, node_feats = load_tg_dataset(DATASET_NAME)
all_event_idxs = [int(v) for v in load_explain_idx(str(EXPLAIN_INDEX_PATH), start=0)]
start = int(EXPLAIN_START)
end = None if N_EVENTS is None else start + int(N_EVENTS)
FORCED_TARGET_EVENT_IDS = list(globals().get("FORCED_TARGET_EVENT_IDS", _forced_target_event_ids_for_combo(DATASET_NAME, MODEL_NAME)))
_target_slice = [int(v) for v in all_event_idxs[start:end]]
_target_budget = len(_target_slice) if N_EVENTS is None else int(N_EVENTS)
target_event_idxs = []
_seen_target_event_idxs = set()
for _event_id in [*FORCED_TARGET_EVENT_IDS, *_target_slice]:
    _event_id = int(_event_id)
    if _event_id in _seen_target_event_idxs:
        continue
    target_event_idxs.append(_event_id)
    _seen_target_event_idxs.add(_event_id)
    if len(target_event_idxs) >= _target_budget:
        break
assert target_event_idxs, "No target events loaded."

ngh_finder = construct_tgat_neighbor_finder(events)

if MODEL_NAME == "tgn":
    mean_time_shift_src, std_time_shift_src, mean_time_shift_dst, std_time_shift_dst = compute_time_statistics(
        events.u.values, events.i.values, events.ts.values
    )

    model = TGN(
        neighbor_finder=ngh_finder,
        node_features=node_feats,
        edge_features=edge_feats,
        device=DEVICE,
        n_layers=p["num_layers"],
        n_heads=p["num_heads"],
        dropout=p["dropout"],
        use_memory=True,
        forbidden_memory_update=True,
        memory_update_at_start=False,
        message_dimension=p["message_dimension"],
        memory_dimension=p["memory_dimension"],
        embedding_module_type="graph_attention",
        message_function="identity",
        mean_time_shift_src=mean_time_shift_src,
        std_time_shift_src=std_time_shift_src,
        mean_time_shift_dst=mean_time_shift_dst,
        std_time_shift_dst=std_time_shift_dst,
        n_neighbors=p["num_neighbors"],
        aggregator_type="last",
        memory_updater_type="gru",
        use_destination_embedding_in_message=False,
        use_source_embedding_in_message=False,
        dyrep=False,
    )
elif MODEL_NAME == "tgat":
    model = TGAN(
        ngh_finder=ngh_finder,
        n_feat=node_feats,
        e_feat=edge_feats,
        device=DEVICE,
        attn_mode=p["attn_mode"],
        use_time=p["use_time"],
        agg_method=p["agg_method"],
        num_layers=p["num_layers"],
        n_head=p["num_heads"],
        num_neighbors=p["num_neighbors"],
        drop_out=p["dropout"],
    )
else:
    raise ValueError(f"Unsupported MODEL_NAME={MODEL_NAME!r}")

state_dict = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state_dict, strict=False)
model.to(DEVICE)
model.eval()

print("Loaded model checkpoint and data.")
print("Events to explain:", len(target_event_idxs))
print("Event range     :", target_event_idxs[0], "->", target_event_idxs[-1])
if len(target_event_idxs) < len(all_event_idxs):
    print(f"Subset run enabled: {len(target_event_idxs)}/{len(all_event_idxs)} events")


#Dataset: simulate_v1, #Users: 2, #Items: 2, #Interactions: 16038, #Timestamps: 16038
#node feats shape: (5, 4), #edge feats shape: (16039, 4)
102 events to explain
Loaded model checkpoint and data.
Events to explain: 30
Event range     : 3 -> 4465
Subset run enabled: 30/102 events


In [5]:
navigator = None
if USE_NAVIGATOR:
    if NAVIGATOR_TYPE == "mlp":
        navigator = MLPNavigator(
            model=model,
            model_name=MODEL_NAME,
            explainer_name="subgraphx_tg",
            dataset_name=DATASET_NAME,
            all_events=events,
            explanation_level=EXPLANATION_LEVEL,
            device=DEVICE,
            results_dir=str(RESULTS_DIR),
            debug_mode=False,
            train_epochs=TRAIN_EPOCHS,
            explainer_ckpt_dir=str(EXPLAINER_CKPT_DIR),
            reg_coefs=REG_COEFS,
            batch_size=BATCH_SIZE,
            lr=LR,
        )
    elif NAVIGATOR_TYPE == "pg":
        navigator = PGNavigator(
            model=model,
            model_name=MODEL_NAME,
            explainer_name="subgraphx_tg",
            dataset_name=DATASET_NAME,
            all_events=events,
            explanation_level=EXPLANATION_LEVEL,
            device=DEVICE,
            results_dir=str(RESULTS_DIR),
            debug_mode=False,
            train_epochs=TRAIN_EPOCHS,
            explainer_ckpt_dir=str(EXPLAINER_CKPT_DIR),
            reg_coefs=REG_COEFS,
            batch_size=BATCH_SIZE,
            lr=LR,
        )
    elif NAVIGATOR_TYPE == "dot":
        navigator = DotProductNavigator(
            model=model,
            model_name=MODEL_NAME,
            device=DEVICE,
            all_events=events,
        )
    else:
        raise ValueError(f"Unsupported NAVIGATOR_TYPE: {NAVIGATOR_TYPE}")

explainer = SubgraphXTG(
    model=model,
    model_name=MODEL_NAME,
    explainer_name="subgraphx_tg",
    dataset_name=DATASET_NAME,
    all_events=events,
    explanation_level=EXPLANATION_LEVEL,
    device=DEVICE,
    results_dir=str(RESULTS_DIR),
    debug_mode=False,
    threshold_num=THRESHOLD_NUM,
    save_results=False,
    mcts_saved_dir=str(MCTS_SAVED_DIR),
    load_results=False,
    rollout=ROLLOUT,
    min_atoms=MIN_ATOMS,
    c_puct=C_PUCT,
    navigator=navigator if USE_NAVIGATOR else None,
    navigator_type=NAVIGATOR_TYPE if USE_NAVIGATOR else None,
    pg_positive=PG_POSITIVE,
)

explain_results = explainer(event_idxs=target_event_idxs)

evaluator = EvaluatorMCTSTG(
    model_name=MODEL_NAME,
    explainer_name="subgraphx_tg",
    dataset_name=DATASET_NAME,
    explainer=explainer,
    results_dir=str(RESULTS_DIR),
    threshold_num=THRESHOLD_NUM,
)

value_results = evaluator.evaluate(explain_results, target_event_idxs)
eval_df = pd.DataFrame(value_results)

eval_csv = evaluator._save_path(
    RESULTS_DIR,
    MODEL_NAME,
    DATASET_NAME,
    "subgraphx_tg",
    target_event_idxs,
    suffix=explainer.suffix,
    threshold_num=THRESHOLD_NUM,
)

print("Saved official evaluator CSV:", eval_csv)
display(eval_df.head())


explainer ckpt loaded from /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/models/tgnnexplainer/simulate_v1/tgn/tgn_simulate_v1_subgraphx_tg_expl_ckpt.pt

explain 0-th: 3
The nodes in graph is 4


mcts simulating: 100%|██████████| 500/500 [00:00<00:00, 1839.11it/s, states=4]


mcts recorder saved at /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_tgnnexplainer/candidate_scores/tgn_simulate_v1_3_mcts_recorder_pg_true_pg_positive_th20.csv

explain 1-th: 92
The nodes in graph is 4


mcts simulating: 100%|██████████| 500/500 [01:06<00:00,  7.48it/s, states=3262]


mcts recorder saved at /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_tgnnexplainer/candidate_scores/tgn_simulate_v1_92_mcts_recorder_pg_true_pg_positive_th20.csv

explain 2-th: 142
The nodes in graph is 4


...

100%|██████████| 3298/3298 [00:00<00:00, 110094.43it/s]



evaluate 29th: 4465


100%|██████████| 3279/3279 [00:00<00:00, 110720.31it/s]

evaluation value results saved at /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_tgnnexplainer/tgn_simulate_v1_subgraphx_tg_3_to_4465_eval_pg_true_pg_positive_th20.csv
Saved official evaluator CSV: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_tgnnexplainer/tgn_simulate_v1_subgraphx_tg_3_to_4465_eval_pg_true_pg_positive_th20.csv


,event_idx,sparsity,fid_inv,fid_inv_best
0,3,0.00,-2.749883,-2.749883
1,3,0.05,-2.749883,-2.749883
2,3,0.10,-2.749883,-2.749883
3,3,0.15,-2.749883,-2.749883
4,3,0.20,-2.749883,-2.749883


In [6]:
# Official TGNNExplainer aggregation (results_processing.ipynb):
# best_fid = max(fid_inv_best), aufsc = trapz(fid_inv_best, sparsity)

from I_explainer_benchmark.src.notebooks.official_explainer_notebook_runtime import aggregate_official_tgnnexplainer_eval

_tgnn_agg = aggregate_official_tgnnexplainer_eval(
    eval_df=eval_df,
    target_event_idxs=target_event_idxs,
    model=model,
    events=events,
    device=DEVICE,
    dataset_name=DATASET_NAME,
    model_name=MODEL_NAME,
    results_dir=RESULTS_DIR,
    navigator_type=NAVIGATOR_TYPE,
    use_navigator=USE_NAVIGATOR,
)

eval_df_official = _tgnn_agg.eval_df_official
summary_official = _tgnn_agg.summary_official
curve_official = _tgnn_agg.curve_official
summary = _tgnn_agg.summary
tab = _tgnn_agg.tab
summary_path = _tgnn_agg.summary_path
curve_path = _tgnn_agg.curve_path
official_curve_csv = _tgnn_agg.official_curve_csv

if len(target_event_idxs) < len(all_event_idxs):
    print(f"Comparability warning: running on {len(target_event_idxs)}/{len(all_event_idxs)} events usually lowers AUFSC/Best FID versus paper-scale runs.")
print("Saved summary:", summary_path)
print("Saved curve  :", curve_path)
print("Note: best_fid uses fid_inv_best (cumulative max), so its argmax is typically sparsity=1.0 by definition.")
display(summary)
display(curve_official[["explainer", "variant", "sparsity", "fid_inv", "fid_inv_best"]])


Comparability warning: running on 30/102 events usually lowers AUFSC/Best FID versus paper-scale runs.
Saved summary: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_tgnnexplainer/tgn_simulate_v1_official_tgnnexplainer_aufsc_bestfid_summary.csv
Saved curve  : /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_tgnnexplainer/tgn_simulate_v1_official_tgnnexplainer_fid_curve.csv
Note: best_fid uses fid_inv_best (cumulative max), so its argmax is typically sparsity=1.0 by definition.


,dataset,model,explainer,navigator,n_events,variant,best_fid,best_fid_sparsity,aufsc,best_minus_aufsc,fid_best_flat_curve,best_fid_raw,best_fid_raw_sparsity,best_fid_raw_lt1,best_fid_raw_lt1_sparsity,flip_success_rate,first_flip_sparsity
0,simulate_v1,tgn,tgnnexplainer,pg,30,official,1.776591,1.0,1.431621,0.34497,False,0.010815,0.95,0.010815,0.95,1.0,0.041667


,explainer,variant,sparsity,fid_inv,fid_inv_best
0,tgnnexplainer,official,0.00,-2.442760,-2.442760
1,tgnnexplainer,official,0.05,-4.323718,-0.854875
2,tgnnexplainer,official,0.10,-5.053526,1.281784
3,tgnnexplainer,official,0.15,-5.559973,1.407847
4,tgnnexplainer,official,0.20,-3.636284,1.439680
5,tgnnexplainer,official,0.25,-2.544749,1.445391
6,tgnnexplainer,official,0.30,-2.524234,1.642602
7,tgnnexplainer,official,0.35,-2.028946,1.642602
8,tgnnexplainer,official,0.40,-0.308191,1.652873
9,tgnnexplainer,official,0.45,-1.014690,1.666093


In [7]:
# Build TGNNExplainer records in the same JSONL schema used by notebook 09 (CoDy metrics path)
from IPython.display import display

from I_explainer_benchmark.src.notebooks.official_explainer_notebook_runtime import build_tgnn_metric_records

_tgnn_records = build_tgnn_metric_records(
    explain_results=explain_results,
    target_event_idxs=target_event_idxs,
    events=events,
    model=model,
)

tgnn_rows = _tgnn_records.tgnn_rows
tgnn_result_records = _tgnn_records.tgnn_result_records
tgnn_results_df = _tgnn_records.tgnn_results_df
display(tgnn_results_df.head())
print("Built TGNNExplainer metric records:", len(tgnn_result_records))


,event_idx,candidate_size,selected_size,selected_event_ids,candidate_event_ids
0,3,2,2,"[1, 2]","[1, 2]"
1,92,20,9,"[77, 80, 81, 83, 84, 85, 87, 88, 91]","[72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 8..."
2,142,20,16,"[122, 124, 126, 127, 129, 130, 131, 132, 133, ...","[122, 123, 124, 125, 126, 127, 128, 129, 130, ..."
3,268,20,3,"[260, 264, 265]","[248, 249, 250, 251, 252, 253, 254, 255, 256, ..."
4,386,20,3,"[373, 381, 384]","[366, 367, 368, 369, 370, 371, 372, 373, 374, ..."


Built TGNNExplainer metric records: 30


In [8]:
# Shared metrics + export pipeline (clean notebook wrapper)
from I_explainer_benchmark.src.notebooks.notebook_metrics_pipeline import run_notebook_metrics_from_namespace

_pipeline_out = run_notebook_metrics_from_namespace(globals(), CFG)
globals().update(_pipeline_out)
run_dir_export = _pipeline_out['run_dir_export']
export_root = _pipeline_out['export_root']
out_jsonl = _pipeline_out['out_jsonl']
out_dir = _pipeline_out['out_dir']
base_name = _pipeline_out['base_name']
print('CURRENT_RUN_ID:', _pipeline_out['CURRENT_RUN_ID'])
print('Saved run export directory:', run_dir_export)
print('Updated global tables under:', export_root)


CURRENT_RUN_ID: simulate_v1_tgn_official_tgnnexplainer_20260315_203348
Saved run export directory: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/summary_ready/simulate_v1_tgn_official_tgnnexplainer_20260315_203348
Updated global tables under: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/summary_ready


In [9]:
# One-spot metrics summary (official)
from I_explainer_benchmark.src.notebooks.notebook_display import show_one_spot_metrics_summary

_one_spot = globals().get("one_spot", globals().get("_pipeline_out", {}).get("one_spot", {}))
show_one_spot_metrics_summary(_one_spot)


One-spot metrics summary (official):


,dataset,model,explainer,variant,n_events,best_fid,best_fid_sparsity,aufsc,best_minus_aufsc,best_fid_raw,...,flip_success_rate,first_flip_sparsity,tempme_acc_auc.ratio_acc,tempme_acc_auc.ratio_prob,tempme_acc_auc.ratio_logit,tempme_acc_auc.ratio_aps,temgx_aufsc,cody_AUFSC_plus,cody_AUFSC_minus,cody_CHARR
0,simulate_v1,tgn,tgnnexplainer,official,30,1.9525300463,0.85,1.4311834499,0.5213465964,1.9525300463,...,0.4666666667,0.0,0.8666666667,0.0675340334,0.5989860823,1.0,0.4150204727,1.2686419564,0.9023438213,1.0545911839
